# NovaPay Day 8: API Packaging & Pipeline

Day 8 packages the NovaPay fraud model behind a deployable FastAPI service. The API validates a single transaction payload, recreates expected engineered features, runs model inference, applies a configurable fraud threshold, and returns a real-time decision for digital money transfer fraud monitoring.

## Deliverables

The project now includes:

```text
api/
  main.py
  schemas.py
  model_loader.py
  scoring.py
  requirements.txt
Dockerfile
reports/artifacts/day8/
  sample_request.json
  sample_response.json
  api_validation_notes.md
```

## Model Loading Check

The API expects the Day 6 model at:

```text
models/day6/best_advanced_model.joblib
```

If this file is unavailable, generate it by running `06_advanced_models.ipynb` first. The `/health` endpoint will still run, but `model_loaded` will be `false`, and `/score` will return a clear `503` message until the model artifact exists.

In [ ]:
from pathlib import Path

model_path = Path("models/day6/best_advanced_model.joblib")
print("Model path:", model_path)
print("Model exists:", model_path.exists())
if not model_path.exists():
    print("Fallback: generate the model by running 06_advanced_models.ipynb first.")

## Expected Request Schema

Required fields: `transaction_id`, `customer_id`, `timestamp`, `home_country`, `source_currency`, `dest_currency`, `channel`, `amount_src`, `amount_usd`, `fee`, `exchange_rate_src_to_dest`, `device_id`, `new_device`, `ip_address`, `ip_country`, `location_mismatch`, `ip_risk_score`, `kyc_tier`, `account_age_days`, `device_trust_score`, `chargeback_history_count`, `risk_score_internal`, `txn_velocity_1h`, `txn_velocity_24h`, and `corridor_risk`.

The API validates non-empty identifiers and categorical fields, valid datetimes, non-negative amounts and counts, positive exchange rates, boolean flags, and bounded risk scores from 0 to 1.

In [ ]:
sample_payload = {
    "transaction_id": "TX12345",
    "customer_id": "CUST1001",
    "timestamp": "2026-07-03T10:15:00Z",
    "home_country": "GB",
    "source_currency": "GBP",
    "dest_currency": "NGN",
    "channel": "mobile_app",
    "amount_src": 750.0,
    "amount_usd": 950.0,
    "fee": 8.5,
    "exchange_rate_src_to_dest": 1850.25,
    "device_id": "DEV-7788",
    "new_device": True,
    "ip_address": "203.0.113.10",
    "ip_country": "NG",
    "location_mismatch": True,
    "ip_risk_score": 0.86,
    "kyc_tier": "tier_2",
    "account_age_days": 24,
    "device_trust_score": 0.22,
    "chargeback_history_count": 1,
    "risk_score_internal": 0.84,
    "txn_velocity_1h": 7,
    "txn_velocity_24h": 28,
    "corridor_risk": 0.78,
}

sample_payload

## Example Invalid Payload

This invalid example shows common validation failures: an empty transaction ID, a negative amount, and a risk score outside the accepted 0-to-1 range.

In [ ]:
invalid_payload = sample_payload.copy()
invalid_payload["transaction_id"] = ""
invalid_payload["amount_usd"] = -25
invalid_payload["ip_risk_score"] = 1.5

invalid_payload

## Run the API Locally

Install dependencies and start the service:

```bash
pip install -r api/requirements.txt
uvicorn api.main:app --reload --host 127.0.0.1 --port 8000
```

Health check URL:

```text
http://127.0.0.1:8000/health
```

In [ ]:
import requests

health_url = "http://127.0.0.1:8000/health"
# Run after starting uvicorn in a terminal.
# requests.get(health_url, timeout=10).json()

## Score One Transaction

The `/score` endpoint accepts one transaction payload and returns the prediction, probability, decision, reason, and model version.

In [ ]:
score_url = "http://127.0.0.1:8000/score"
# Run after starting uvicorn and ensuring the Day 6 model exists.
# response = requests.post(score_url, json=sample_payload, timeout=10)
# response.status_code, response.json()

## Example Response Format

```json
{
  "transaction_id": "TX12345",
  "prediction": "fraud",
  "fraud_probability": 0.92,
  "confidence_score": 0.92,
  "decision": "hold_and_investigate",
  "reason": "Transaction flagged for high transaction velocity, high IP risk score, low device trust score, location mismatch, previous chargeback history, high corridor risk, unusual high internal risk score.",
  "model_version": "day6_best_advanced_model"
}
```

## Docker Commands

Build the image:

```bash
docker build -t novapay-fraud-api .
```

Run the container:

```bash
docker run -p 8000:8000 novapay-fraud-api
```

Check the running API:

```text
http://127.0.0.1:8000/health
```

## Reflection

Which input validations are critical during real-time scoring?

Critical validations include ensuring non-negative transaction amounts, valid timestamps, non-empty identifiers, valid country/currency fields, correct boolean values, and bounded risk scores. These checks prevent invalid or malicious payloads from entering the model pipeline, reduce scoring errors, and improve trust in real-time fraud decisions.

## Assessment

Sample API request: `reports/artifacts/day8/sample_request.json`

Sample API response: `reports/artifacts/day8/sample_response.json`

The request demonstrates a high-risk transaction, and the response demonstrates the expected fraud decision structure returned by the deployed service.